# Caspase-1 activity — rhein dose response

Cell-free colorimetric caspase-1 assay (Enzo BML-AK701, YVAD-pNA substrate, read at 405 nm).
Rhein at 10, 20, 40 and 60 µM against a shared vehicle control, with the selective inhibitor
VRT-043198 as positive control.

**Pipeline**

1. Read raw absorbances from `Caspae-1 Activity.xlsx` (sheet `Planilha1`, one column pair per group:
   raw OD and the workbook's own percentage column).
2. Express every well as a percentage of the **mean** control OD, so the control group averages 100
   by construction and the reference line at 100 is the group the tests compare against. The
   workbook's own `%` columns divide by a single control well instead; set `NORMALISE = 'first_well'`
   to reproduce them.
3. One-way ANOVA across control and the four rhein concentrations, then **Dunnett's test** of each
   concentration against the shared control — the correct multiplicity frame for several doses
   against one control, and narrower than correcting for all pairwise comparisons. Welch's
   unequal-variance t-tests are computed alongside as a sensitivity check.
4. VRT-043198 is tested against control separately; it is an assay control, not part of the dose
   response, and is excluded from the ANOVA.
5. Figure at 4.1 × 5.0 cm with 8 pt type, matching the published panel's geometry.

> **Provenance warning — read before using the output of this notebook.**
> The figure this notebook draws does **not** match the previously circulated PDF
> (`caspase1_rhein_dose_4p1.pdf`). Measuring the bar heights out of that PDF's vector content gives
> 100.0, 92.3, 87.3, 79.8 and 69.9 % — a clean monotonic dose response. This workbook gives
> 100.0, 97.7, 88.9, 91.3 and 88.1 %, which is not monotonic: 40 µM sits above 20 µM.
> A third version of the same experiment exists in `Caspae-1 Activity.pzfx` (GraphPad Prism 6,
> authored June 2023); its stored values are monotonic and reproduce four of the five published
> bars but give 79.4 % at 60 µM rather than 69.9 %. The published 60 µM bar equals, to four
> significant figures, the ratio of the two bars in the separate `caspase1_rhein60_plate.pdf`
> experiment (62.43 / 89.27 = 69.93 %), so the two figures are not independent.
> Which dataset is correct is a question for the bench records. This notebook analyses the raw
> absorbances in the workbook and reports what they contain.

In [ ]:
#@title 1 · Upload the workbook (skip if you mounted Drive)
try:
    import google.colab, os
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

XLSX = 'Caspae-1 Activity.xlsx'

if IN_COLAB:
    import os
    if not os.path.exists(XLSX):
        from google.colab import files
        up = files.upload(); XLSX = list(up.keys())[0]
print('Workbook:', XLSX)

In [ ]:
#@title 2 · Setup and configuration
import numpy as np, pandas as pd, openpyxl
import matplotlib as mpl, matplotlib.pyplot as plt
from scipy import stats

SHEET      = 'Planilha1'
# group -> 0-based column of the raw OD in the sheet (the '%' column is the one after it)
COLUMNS    = {'Control': 0, 'Rhein 10': 2, 'Rhein 20': 4, 'Rhein 40': 6, 'Rhein 60': 8,
              'VRT-043198': 10}
DOSE_ORDER = ['Control', 'Rhein 10', 'Rhein 20', 'Rhein 40', 'Rhein 60']
POSITIVE   = 'VRT-043198'
XLABELS    = ['Control', '10', '20', '40', '60']
NORMALISE  = 'control_mean'   # 'control_mean' or 'first_well' (what the workbook's % columns do)
ALPHA      = 0.05
SAVE_PREFIX = 'caspase1'
print('ok')

In [ ]:
#@title 3 · Read the absorbances
ws = openpyxl.load_workbook(XLSX, data_only=True)[SHEET]
rows = [r for r in ws.iter_rows(min_row=2, values_only=True) if any(v is not None for v in r)]

od = {}
for g, c in COLUMNS.items():
    od[g] = np.array([r[c] for r in rows if len(r) > c and r[c] is not None], dtype=float)

ref = od['Control'].mean() if NORMALISE == 'control_mean' else od['Control'][0]
pct = {g: v / ref * 100 for g, v in od.items()}

print(f'normalised to {NORMALISE} (OD = {ref:.5f})\n')
raw = pd.DataFrame([dict(group=g, n=len(v), od_mean=v.mean(), od_sd=v.std(ddof=1),
                         pct_mean=pct[g].mean(), pct_sd=pct[g].std(ddof=1))
                    for g, v in od.items()])
display(raw.round(3))

In [ ]:
#@title 4 · Statistics — one-way ANOVA, Dunnett vs the shared control
groups = [pct[g] for g in DOSE_ORDER]
F, p_anova = stats.f_oneway(*groups)
print(f'One-way ANOVA across {", ".join(DOSE_ORDER)}: F({len(DOSE_ORDER)-1},'
      f'{sum(len(g) for g in groups) - len(DOSE_ORDER)}) = {F:.3f}, p = {p_anova:.4g}\n')

try:
    dun = stats.dunnett(*groups[1:], control=groups[0])
    p_dunnett = list(dun.pvalue)
    test_name = "Dunnett's test vs control"
except AttributeError:                      # scipy < 1.11
    p_dunnett = [np.nan] * (len(DOSE_ORDER) - 1)
    test_name = "Dunnett unavailable (scipy < 1.11)"

rows_out = []
for g, pd_ in zip(DOSE_ORDER[1:], p_dunnett):
    w = stats.ttest_ind(pct[g], pct['Control'], equal_var=False)
    n1, n2 = len(pct['Control']), len(pct[g])
    sp = np.sqrt(((n1-1)*pct['Control'].var(ddof=1) + (n2-1)*pct[g].var(ddof=1)) / (n1+n2-2))
    d  = (pct[g].mean() - pct['Control'].mean()) / sp if sp > 0 else np.nan
    rows_out.append(dict(group=g, n=n2, pct_mean=pct[g].mean(), pct_sd=pct[g].std(ddof=1),
                         diff_vs_control=pct[g].mean() - pct['Control'].mean(),
                         p_dunnett=pd_, p_welch=w.pvalue,
                         hedges_g=d * (1 - 3 / (4*(n1+n2) - 9))))
res = pd.DataFrame(rows_out)
display(res.round(4))

pv = stats.ttest_ind(pct[POSITIVE], pct['Control'], equal_var=False)
print(f'{POSITIVE} vs control (assay control, not in the ANOVA): '
      f'{pct[POSITIVE].mean():.1f}% of control, Welch p = {pv.pvalue:.3g}')

In [ ]:
#@title 5 · Figure — 4.1 x 5.0 cm, 8 pt
CM = 1 / 2.54
AHA = {'font.family': 'sans-serif',
       'font.sans-serif': ['Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans'],
       'font.size': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
       'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6,
       'xtick.major.size': 2.5, 'ytick.major.size': 2.5, 'xtick.direction': 'out',
       'pdf.fonttype': 42, 'ps.fonttype': 42, 'savefig.dpi': 600, 'savefig.bbox': 'standard'}

PANEL_W_CM, PANEL_H_CM = 4.1, 5.0
MARGIN   = dict(l=1.58, r=0.12, b=1.78, t=0.20)   # cm
BAR_FACE = '#C8C8C8'
Y_MAX    = 125
ANNOTATE = 'p_dunnett'        # or 'p_welch'
PANEL_LABEL = None            # e.g. 'A' if this becomes a lettered panel

def stars(p):
    if not np.isfinite(p): return ''
    return '****' if p < 1e-4 else '***' if p < 1e-3 else '**' if p < 1e-2 else '*' if p < 0.05 else ''

def fig_dose():
    with mpl.rc_context(AHA):
        m = MARGIN
        fig = plt.figure(figsize=(PANEL_W_CM * CM, PANEL_H_CM * CM))
        ax  = fig.add_axes([m['l'] / PANEL_W_CM, m['b'] / PANEL_H_CM,
                            1 - (m['l'] + m['r']) / PANEL_W_CM,
                            1 - (m['b'] + m['t']) / PANEL_H_CM])
        rng = np.random.default_rng(7)
        x   = np.arange(len(DOSE_ORDER))
        for i, g in enumerate(DOSE_ORDER):
            v = pct[g]
            ax.bar(i, v.mean(), width=0.68, facecolor=BAR_FACE, edgecolor='none', zorder=2)
            ax.scatter(rng.normal(i, 0.07, len(v)), v, s=6, color='black', zorder=4, clip_on=False)
            if g != 'Control':
                lab = stars(float(res.loc[res.group == g, ANNOTATE].iloc[0]))
                if lab:
                    ax.text(i, Y_MAX * 0.94, lab, ha='center', va='center', fontsize=8)
        ax.axhline(100, ls='--', color='k', lw=0.8, zorder=1)
        ax.set_xticks(x); ax.set_xticklabels(XLABELS, rotation=45, ha='right', rotation_mode='anchor')
        ax.set_xlim(-0.65, len(DOSE_ORDER) - 0.35)
        ax.set_ylim(0, Y_MAX); ax.set_yticks([0, 25, 50, 75, 100, 125])
        ax.set_ylabel('Caspase-1 activity\n(% of control)')
        ax.set_xlabel('Rhein (µM)', labelpad=2)
        ax.spines[['top', 'right']].set_visible(False)
        if PANEL_LABEL:
            fig.text(0.015, 0.985, PANEL_LABEL, ha='left', va='top', fontsize=10, fontweight='bold')
    return fig

fig = fig_dose()
with mpl.rc_context(AHA):
    fig.savefig(f'{SAVE_PREFIX}_rhein_dose.pdf')
    fig.savefig(f'{SAVE_PREFIX}_rhein_dose.tiff', dpi=600, pil_kwargs={'compression': 'tiff_lzw'})
    fig.savefig(f'{SAVE_PREFIX}_rhein_dose.png', dpi=600)
w, h = fig.get_size_inches()
print(f'Panel size: {w/CM:.2f} x {h/CM:.2f} cm')
plt.show()

In [ ]:
#@title 6 · Written report and tables
L = []
L.append('CASPASE-1 ACTIVITY - RHEIN DOSE RESPONSE')
L.append('=' * 72)
L.append(f'Assay: cell-free colorimetric caspase-1 (YVAD-pNA), absorbance at 405 nm.')
L.append(f'Normalisation: each well as a percentage of the {NORMALISE.replace("_", " ")} '
         f'(OD = {ref:.5f}).')
L.append(f'Primary test: one-way ANOVA across control and four concentrations, then Dunnett '
         f'against the shared control. Welch t-tests reported alongside. Alpha = {ALPHA}.')
L.append('')
L.append(f'One-way ANOVA: F = {F:.3f}, p = {p_anova:.4g}')
L.append('')
for _, r in res.iterrows():
    L.append(f'{r.group:12s} n = {int(r.n)}  {r.pct_mean:6.2f} +/- {r.pct_sd:5.2f} % of control '
             f'({r.diff_vs_control:+.2f} pp)')
    L.append(f'               Dunnett p = {r.p_dunnett:.4f}   Welch p = {r.p_welch:.4f}   '
             f'Hedges g = {r.hedges_g:+.2f}')
L.append('')
L.append(f'{POSITIVE} (assay control): {pct[POSITIVE].mean():.2f} +/- '
         f'{pct[POSITIVE].std(ddof=1):.2f} % of control, Welch p = {pv.pvalue:.3g}')
report = '\n'.join(L)
print(report)

with open(f'{SAVE_PREFIX}_stats_report.txt', 'w') as f:
    f.write(report)
res.to_csv(f'{SAVE_PREFIX}_stats_table.csv', index=False)
pd.DataFrame({g: pd.Series(v) for g, v in pct.items()}).to_csv(
    f'{SAVE_PREFIX}_percent_of_control.csv', index=False)
print('\nSaved: report, stats table, per-well percentages.')